# A Comparative Analysis of Optimization Trajectories and Convergence Properties

> **Estimated Runtime Without Cache on 7900XTX**
> | Step                       | Time     |
> |----------------------------|----------|
> | Grid Search                | ~46 mins |
> | Trajectory analysis        | ~68 mins |
> | Convergence point anaylsis | ~20 mins |

> **Estimated Runtime With Cache on 7900XTX**
> | Step                       | Time    |
> |----------------------------|---------|
> | Grid Search                | <1 mins  |
> | Trajectory analysis        | <1 mins  |
> | Convergence point anaylsis | ~20 mins |

In [ ]:
USE_CACHED=True

---
## Environment & Setup

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import pickle
from tqdm.auto import tqdm

import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as Transforms
from lion_pytorch import Lion

---

## Optimizers, Model, and Dataset

In [ ]:
class MuonSGD:
    def __init__(self, muon_params, sgd_params, lr):
        self.muon = optim.Muon(muon_params, lr=lr) if muon_params else None
        self.sgd = optim.SGD(sgd_params, lr=lr) if sgd_params else None
    def zero_grad(self):
        if self.muon:
            self.muon.zero_grad()
        if self.sgd:
            self.sgd.zero_grad()
    def step(self):
        if self.muon:
            self.muon.step()
        if self.sgd:
            self.sgd.step()


def get_optimizer(params, name, lr):
    name = name.lower()
    if name == "sgd":
        return optim.SGD(params, lr=lr)
    if name == "adam":
        return optim.Adam(params, lr=lr)
    if name == "lion":
        return Lion(params, lr=lr)
    if name == "muon":
        params = list(params)
        muon_params = [p for p in params if p.ndim == 2]
        sgd_params = [p for p in params if p.ndim != 2]
        return MuonSGD(muon_params, sgd_params, lr)
    raise ValueError(f"Unknown optimizer: {name}")

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes=(256, 128)):
        super().__init__()
        layers = [nn.Flatten(), nn.Linear(28 * 28, hidden_sizes[0]), nn.ReLU()]
        for i in range(len(hidden_sizes) - 1):
            layers += [nn.Linear(hidden_sizes[i], hidden_sizes[i + 1]), nn.ReLU()]
        layers.append(nn.Linear(hidden_sizes[-1], 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def get_loaders(batch_size):
    transform = Transforms.Compose([Transforms.ToTensor(), Transforms.Normalize((0.5,), (0.5,))])
    train = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
    test = datasets.FashionMNIST("data", train=False, download=True, transform=transform)
    pin = torch.cuda.is_available()
    return (
        DataLoader(train, batch_size=batch_size, shuffle=True, num_workers=1, pin_memory=pin),
        DataLoader(test, batch_size=batch_size, shuffle=False, num_workers=1, pin_memory=pin),
    )

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total, loss_sum = 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += y.size(0)
        loss_sum += loss.item() * y.size(0)
    return loss_sum / total


def eval_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.size(0)
    return correct / total

In [ ]:
def train(opt, batch_size, lr, epochs, device):
    model = MLP().to(device)
    train_loader, test_loader = get_loaders(batch_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model.parameters(), opt, lr)

    for epoch in tqdm(range(epochs)):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        test_acc = eval_model(model, test_loader, device)
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f} - Test Acc: {test_acc:.4f}")

---

## Grid Search for parmeters

In [ ]:
epochs = 20
learning_rates = [1e-4, 1e-3, 1e-2]
batch_sizes = [32, 64, 128]
optimizers_to_test = ["sgd", "adam", "lion", "muon"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
def grid_search(learning_rates, batch_sizes, optimizers, epochs, device):
    results = {}
    total_runs = len(learning_rates) * len(batch_sizes) * len(optimizers)
    current_run = 0
    
    for opt in optimizers:
        for lr in learning_rates:
            for bs in batch_sizes:
                current_run += 1
                print(f"\n[{current_run}/{total_runs}] Training {opt.upper()} with lr={lr:.0e}, batch_size={bs}")
                
                model = MLP().to(device)
                train_loader, test_loader = get_loaders(bs)
                criterion = nn.CrossEntropyLoss()
                optimizer = get_optimizer(model.parameters(), opt, lr)
                
                best_loss = float('inf')
                for epoch in tqdm(range(epochs), leave=False):
                    epoch_loss = train_epoch(model, train_loader, criterion, optimizer, device)
                    best_loss = min(best_loss, epoch_loss)
                
                final_acc = eval_model(model, test_loader, device)
                results[(opt, lr, bs)] = (best_loss, final_acc)
                print(f"  Best Loss: {best_loss:.4f}, Final Acc: {final_acc:.4f}")
    
    return results

def compute_best_hyprparams(results):
    best_hyperparams = {}
    best_metrics = {}
    for (opt, lr, bs), (best_loss, final_acc) in results.items():
        current_score = (final_acc, -best_loss)
        if opt not in best_metrics or current_score > best_metrics[opt]:
            best_metrics[opt] = current_score
            best_hyperparams[opt] = {"lr": lr, "bs": bs}
    return best_hyperparams

if USE_CACHED:
    best_hyperparams = {
        "sgd": {"lr": 1e-2, "bs": 32},
        "adam": {"lr": 1e-3, "bs": 64},
        "lion": {"lr": 1e-4, "bs": 128},
        "muon": {"lr": 1e-3, "bs": 64},
    }
else:
    results = grid_search(learning_rates, batch_sizes, optimizers_to_test, epochs, device)
    best_hyperparams = compute_best_hyprparams(results)

---

## Trajectory analysis

In [ ]:
def get_model_params(model):
    params = []
    for p in model.parameters():
        params.append(p.data.cpu().clone().flatten())
    return torch.cat(params)


def set_model_params(model, params_flat):
    offset = 0
    for p in model.parameters():
        numel = p.data.numel()
        p.data = params_flat[offset:offset + numel].reshape(p.data.shape).to(p.device)
        offset += numel


def compute_loss_at_interpolation(model, loader, criterion, theta_start, theta_end, device, num_points):
    model.eval()
    alphas = np.linspace(0, 1, num_points)
    losses = []
    
    with torch.no_grad():
        for alpha in alphas:
            theta_alpha = (1 - alpha) * theta_start + alpha * theta_end
            set_model_params(model, theta_alpha)
            
            total_loss = 0.0
            total_samples = 0
            for x, y in loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                loss = criterion(logits, y)
                total_loss += loss.item() * y.size(0)
                total_samples += y.size(0)
            
            losses.append(total_loss / total_samples)
    
    return alphas, np.array(losses)


def train_with_epoch_trajectories(opt, lr, epochs, device, train_loader, test_loader, criterion):
    model = MLP().to(device)
    optimizer = get_optimizer(model.parameters(), opt, lr)
    
    trajectories = []
    train_losses = []
    
    for epoch in tqdm(range(epochs), desc=f"{opt.upper()}"):
        # Save parameters at epoch start
        theta_epoch_start = get_model_params(model).to(device)
        
        model.train()
        total, loss_sum = 0, 0.0
        
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total += y.size(0)
            loss_sum += loss.item() * y.size(0)
        
        epoch_loss = loss_sum / total
        train_losses.append(epoch_loss)
        
        # Save parameters at epoch end and compute trajectory
        theta_epoch_end = get_model_params(model).to(device)
        alphas, losses = compute_loss_at_interpolation(model, train_loader, criterion, theta_epoch_start, theta_epoch_end, device, num_points=50)
        trajectories.append((epoch, alphas, losses))
        
        # Evaluate on test set
        test_acc = eval_model(model, test_loader, device)
        print(f"  Epoch {epoch+1}/{epochs} - Train Loss: {epoch_loss:.4f} - Test Acc: {test_acc:.4f}")
    
    return model, trajectories, train_losses

In [ ]:
TRAJECTORY_CACHE="data/trajectories_cache.pkl"
def save_trajectories(trajectories_dict, filename=TRAJECTORY_CACHE):
    with open(filename, 'wb') as f:
        pickle.dump(trajectories_dict, f)

def load_trajectories(filename=TRAJECTORY_CACHE):
    if os.path.exists(filename):
        with open(filename, 'rb') as f:
            data = pickle.load(f)
        return data
    return None

In [ ]:
if USE_CACHED:
    trajectories_all_optimizers = load_trajectories()
else:
    epochs = 5
    trajectories_all_optimizers = {}

    for opt in optimizers_to_test:
        params = best_hyperparams[opt]
        lr = params["lr"]
        bs = params["bs"]
        
        print(f"\n{opt.upper()}: lr={lr:.0e}, batch_size={bs}")
        print("-" * 40)
        
        train_loader, test_loader = get_loaders(bs)
        criterion = nn.CrossEntropyLoss()
        
        model, trajectories, train_losses = train_with_epoch_trajectories(opt, lr, epochs, device, train_loader, test_loader, criterion)
        trajectories_all_optimizers[opt] = trajectories
    
    save_trajectories(trajectories_all_optimizers)

In [ ]:
# Enable LaTeX rendering
plt.rcParams['text.usetex'] = True
plt.rcParams['font.size'] = 11

optimizer_colors = {'sgd': 'C0', 'adam': 'C1', 'lion': 'C2', 'muon': 'C3'}

# Create one plot per epoch with all optimizers
for epoch_idx in range(5):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for opt in optimizers_to_test:
        trajectories = trajectories_all_optimizers[opt]
        if epoch_idx < len(trajectories):
            epoch, alphas, losses = trajectories[epoch_idx]
            
            ax.plot(alphas, losses, linewidth=2.5, marker='o', markersize=5,
                   label=opt.upper(), color=optimizer_colors[opt])
            
            convex_line = (1 - alphas) * losses[0] + alphas * losses[-1]
            ax.plot(alphas, convex_line, linewidth=1.5, linestyle='--', 
                   color=optimizer_colors[opt], alpha=0.3)
    
    ax.set_xlabel(r'Interpolation parameter $\alpha$', fontsize=12)
    ax.set_ylabel(r'Loss', fontsize=12)
    ax.set_title(f'Loss Trajectories --- Epoch {epoch_idx+1}', fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=11, framealpha=0.9)
    
    fig.tight_layout()
    plt.savefig(f'loss_trajectory_epoch{epoch_idx+1}.pdf', dpi=150, bbox_inches='tight')
    plt.show()

### Smoothness over trajectory

In [ ]:
for opt in optimizers_to_test:
    trajectories = trajectories_all_optimizers[opt]
    
    print(f"\n{opt.upper()}:")
    print("-" * 40)
    
    smoothness_scores = []
    for epoch, alphas, losses in trajectories:
        # ||f'(x_i) - f'(x_{i+1})|| <= L ||x_i - x_{i+1}|| is the bound.
        slopes = np.diff(losses) / np.diff(alphas)
        curvature = np.diff(slopes) / np.diff(alphas[:-1])
        smoothness = np.max(np.abs(curvature))
        smoothness_scores.append(smoothness)
        print(f"  Epoch {epoch+1}: Smoothness: {smoothness:.4f}")
    
    avg_smoothness = np.mean(smoothness_scores)
    print(f"\n  Summary:")
    print(f"    Avg smoothness: {avg_smoothness:.4f}")

---

## Convergence point analysis

In [ ]:
def compute_gradient_flat(model, loader, criterion, device):
    model.train()
    model.zero_grad()
    total_loss, total_samples = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        loss = criterion(model(x), y) * y.size(0)
        loss.backward()
        total_loss += loss.item()
        total_samples += y.size(0)
    grads = [p.grad.data.cpu().clone().flatten() / total_samples for p in model.parameters()]
    return torch.cat(grads)


def compute_loss_flat(model, loader, criterion, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            total_loss += criterion(model(x), y).item() * y.size(0)
            total_samples += y.size(0)
    return total_loss / total_samples


def test_convergence_condition(model, loader, criterion, device, num_samples):
    theta_m = get_model_params(model).cpu()
    f_m = compute_loss_flat(model, loader, criterion, device)
    grad_m = compute_gradient_flat(model, loader, criterion, device)
    print(f"epsilon = {10 * grad_m.norm().item()}")
    epsilon = 10 * grad_m.norm().item()

    violations = 0
    for _ in range(num_samples):
        direction = torch.randn_like(theta_m)
        theta_x = theta_m + epsilon * direction / direction.norm()

        set_model_params(model, theta_x.to(device))
        f_x = compute_loss_flat(model, loader, criterion, device)

        rhs = f_x + torch.dot(grad_m, theta_m - theta_x).item()
        if f_m > rhs + 1e-6:
            violations += 1

    set_model_params(model, theta_m.to(device))
    return violations, num_samples


epoch = 10
for opt in optimizers_to_test:
    params = best_hyperparams[opt]
    train_loader, _ = get_loaders(params["bs"])
    criterion = nn.CrossEntropyLoss()

    model = MLP().to(device)
    optimizer = get_optimizer(model.parameters(), opt, params["lr"])
    for _ in tqdm(range(epoch), desc=f"Training {opt.upper()}"):
        train_epoch(model, train_loader, criterion, optimizer, device)

    violations, total = test_convergence_condition(model, train_loader, criterion, device, 50)
    pct = (total - violations) / total * 100
    print(f"{opt.upper():6s}: {violations}/{total} violations  ({pct:.1f}% condition satisfied)")